# Fit Model on full Dev Set

- Use model config derrived for each of self/parent cohorts in exploration to fit a single model
- Extract coefs from each trained self/parent model

## Set Up

In [ ]:
import sys, os

sys.path.append(os.path.abspath("../"))

from src.config import BASE_PATH, SEED
from src.utils import col_config_sensitivity
from src.coef_visual import plot_heatmap
from src.residual_func import (
    get_perm_feat_imp,
    build_model,
    get_saliency_map,
)

In [ ]:
shared_dict = {
    "num_layers": 3,
    "max_iter": 50000,
    "seed": SEED,
    "linear_type": "ridge",
}
parent_config = {
    "impact_type": "parent",
    "quest_cols": col_config_sensitivity["parent_quest"],
    "agg_type": "max",
    "alpha2": 10.0,
    "alpha3": 5.0,
    "desc_seg_type": "pt_noshort",
    **shared_dict,
}
self_config = {
    "impact_type": "self",
    "quest_cols": col_config_sensitivity["self_quest"],
    "agg_type": "block",
    "alpha2": 40.0,
    "alpha3": 0.5,
    "desc_seg_type": "turns",
    **shared_dict,
}

## Build Model

In [ ]:
data_dir = BASE_PATH / "data"
llama_model = "Llama-3.3-70B-Instruct"
model_parent, coef_parent, df_parent = build_model(parent_config, llama_model, data_dir)
model_self, coef_self, df_self = build_model(self_config, llama_model, data_dir)

In [ ]:
coef_parent = coef_parent.rename(columns={"coef": "coef_parent"})
coef_self = coef_self.rename(columns={"coef": "coef_self"})
coef_combined = coef_parent.join(coef_self)
plot_heatmap(
    coef_combined,
    title="Learned Coef weights for final parent and self models",
    vmin=-2.5,
    vmax=2.5,
    xlabel="Cohort",
)

# Saliency Sanity Checks

In [ ]:
saliency_self = get_saliency_map(self_config, model_self, df_self, coef_self)

In [ ]:
saliency_parent = get_saliency_map(parent_config, model_parent, df_parent, coef_parent)

# LMG Feat Imp

- Randomly sample order of text outcomes n times
- Iteratively compute incremental r^2 after adding each outcome
- Avg across all 1000 samples to get avg contribution to accuracy

**TODO:**
- Implement via CV instead of single train-test split

In [ ]:
res_df = get_perm_feat_imp(
    train_df=df_parent,
    test_df=df_parent,
    model_config_dict=parent_config,
    n_samples=500,
    random_state=SEED,
    n_jobs=2,
)
res_df